# Import libraries

In [ ]:
import torch
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.model_selection import KFold
import importlib
import MLUtils
importlib.reload(MLUtils)
from MLUtils import GPR, trainTestData

# Project directories

In [ ]:
currentDir = Path.cwd().resolve()

projectDir = next(
    (path for path in [currentDir, *currentDir.parents] if (path / '.git').exists()),
    None
)

# Set up training data

In [ ]:
# CFD data import
trainingDataPath = projectDir/'ML'/'data'/'trainingData.txt'
trainingData = pd.read_csv(trainingDataPath, sep = '\t')

# maximum Cd and Cl variance for noise constraint
CdMaxVar = np.max((trainingData['CdCoV']*trainingData['Cd'])**2)
ClMaxVar = np.max((trainingData['ClCoV']*trainingData['Cl'])**2)

noiseFloorDict = {
    'Cd' : CdMaxVar,
    'Cl' : ClMaxVar
}

# independent variables
X = trainingData[['M', 'P', 'T', 'AoA']].to_numpy()

isSymmetric = np.zeros((len(trainingData), 1))  # add a binary toggle separate symmetric and cambered domain
isSymmetric[X[:, 0] == 0, 0] = 1
X = np.column_stack([X, isSymmetric])

X[X[:, 0] == 0, 1] = 4  # change the placeholder value of P to 4 when m = 0

# dependent variables
yCd = trainingData['Cd'].to_numpy()
yCl = trainingData['Cl'].to_numpy()

# k-fold generation
randomState = 1

while True:  # ensure each test split has at leasy 1 point in the symmetric domain
    kf = KFold(n_splits = 5, shuffle = True, random_state = randomState)

    folds = [(trainIdx, testIdx) for trainIdx, testIdx in kf.split(X)]

    valid = True

    for foldI in range (5):
        _, testIdx = folds[foldI]
        XTest = X[testIdx]
        XSTest = XTest[XTest[:, 0] == 0]

        if len(XSTest) == 0:
            randomState += 1
            valid = False
            break

    if valid:
        break

# bounds of independent variables for normalising
XMin = torch.tensor([0, 2, 6, 0, 0], dtype = torch.float64)
XMax = torch.tensor([6, 6, 24, 8, 1], dtype = torch.float64)

In [ ]:
print (noiseFloorDict)

# Cross-validation

In [ ]:
# set up dictionary for information storage
MLValResultsPath = projectDir/'ML'/'data'/'MLValResults.txt'
MLValResultsPath.parent.mkdir(parents=True, exist_ok=True)

MLValResults = {
    'depVar' : [], 
    'foldI' : [], 
    'kernel' : [], 
    'ARD' : [],
    'trainRMSE' : [], 
    'testSRMSE' : [], 
    'testCRMSE' : [], 
    'trainMAE' : [], 
    'testSMAE' : [],
    'testCMAE' : [],
    'trainNLPD' : [], 
    'testSNLPD' : [],
    'testCNLPD' : [],
    'noise' : []
}

# generate combinations of ARDs and kernels for each dependent variable and fold
for depVar in ['Cd', 'Cl']:
    for ARD in [None, 5]:
        for kernel in ['rbf', 'matern32', 'matern52']:
            for foldI in range (0, 5):
                MLValResults['depVar'].append(depVar)
                MLValResults['foldI'].append(foldI)
                MLValResults['kernel'].append(kernel)
                MLValResults['ARD'].append(ARD)

# evalute training and test scores for each set up combination
for i in range (len(MLValResults['depVar'])):
    # training and test information
    depVar = MLValResults['depVar'][i]
    foldI = MLValResults['foldI'][i]
    kernel = MLValResults['kernel'][i]
    ARD = MLValResults['ARD'][i]
    noiseFloor = noiseFloorDict[depVar]
    XTrain, yTrain, XSTest, ySTest, XCTest, yCTest = trainTestData (X, yCd, yCl, folds, depVar, foldI, XMin, XMax)

    # evaluate and obtain test scores
    trainRMSE, testSRMSE, testCRMSE, trainMAE, testSMAE, testCMAE, trainNLPD, testSNLPD, testCNLPD, noise = GPR(kernel, ARD, noiseFloor,
                                                                                                                XTrain, yTrain, XSTest, ySTest, XCTest, yCTest)
    MLValResults['trainRMSE'].append(trainRMSE)
    MLValResults['testSRMSE'].append(testSRMSE)
    MLValResults['testCRMSE'].append(testCRMSE)
    MLValResults['trainMAE'].append(trainMAE)
    MLValResults['testSMAE'].append(testSMAE)
    MLValResults['testCMAE'].append(testCMAE)
    MLValResults['trainNLPD'].append(trainNLPD)
    MLValResults['testSNLPD'].append(testSNLPD)
    MLValResults['testCNLPD'].append(testCNLPD)
    MLValResults['noise'].append(noise)

# save results
MLValResults = pd.DataFrame(MLValResults)
MLValResults.to_csv(MLValResultsPath, sep="\t", index=False)


# Held-out test

In [ ]:
# set up training and testing information

XTrain = (torch.tensor(X, dtype = torch.float64) - XMin)/(XMax - XMin)  # normalise independent variables

yCdTrain = torch.tensor(yCd, dtype = torch.float64)
yClTrain = torch.tensor(yCl, dtype = torch.float64)

testDataPath = projectDir/'ML'/'data'/'testData.txt'
testData = pd.read_csv(testDataPath, sep = '\t')

yCdTest = testData['Cd'].to_numpy()
yClTest = testData['Cl'].to_numpy()

XTest = testData[['M', 'P', 'T', 'AoA']].to_numpy()

isSymmetric = np.zeros((len(testData), 1))  # add a binary toggle separate symmetric and cambered domain
isSymmetric[XTest[:, 0] == 0, 0] = 1
XTest = np.column_stack([XTest, isSymmetric])

XTest[XTest[:, 0] == 0, 1] = 4  # change the placeholder value of P to 4 when m = 0

# separating into symmetric and cambered data sets
XSTest = (torch.tensor(XTest[XTest[:, 0] == 0], dtype = torch.float64) - XMin)/(XMax - XMin)
XCTest = (torch.tensor(XTest[XTest[:, 0] > 0], dtype = torch.float64) - XMin)/(XMax - XMin)

yCdSTest = torch.tensor(yCdTest[XTest[:, 0] == 0], dtype = torch.float64)
yCdCTest = torch.tensor(yCdTest[XTest[:, 0] > 0], dtype = torch.float64)

yClSTest = torch.tensor(yClTest[XTest[:, 0] == 0], dtype = torch.float64)
yClCTest = torch.tensor(yClTest[XTest[:, 0] > 0], dtype = torch.float64)

# prepare dictionary for information storage
MLTestResultsPath = projectDir/'ML'/'data'/'MLTestResults.txt'
MLTestResultsPath.parent.mkdir(parents=True, exist_ok=True)

MLTestResults = {
    'depVar' : ['Cd', 'Cl'], 
    'trainRMSE' : [], 
    'testSRMSE' : [], 
    'testCRMSE' : [], 
    'trainMAE' : [], 
    'testSMAE' : [],
    'testCMAE' : [],
    'trainNLPD' : [], 
    'testSNLPD' : [],
    'testCNLPD' : [],
    'noise' : []
}

# selected kernel and ARD
kernel = 'matern52'
ARD = 5

# evaluate and obtain test scores
for depVar, yTrain, ySTest, yCTest in [('Cd', yCdTrain, yCdSTest, yCdCTest), ('Cl', yClTrain, yClSTest, yClCTest)]:
    noiseFloor = noiseFloorDict[depVar]
    trainRMSE, testSRMSE, testCRMSE, trainMAE, testSMAE, testCMAE, trainNLPD, testSNLPD, testCNLPD, noise = GPR(kernel, ARD, noiseFloor, 
                                                                                                                XTrain, yTrain, XSTest, ySTest, XCTest, yCTest)
    MLTestResults['trainRMSE'].append(trainRMSE)
    MLTestResults['testSRMSE'].append(testSRMSE)
    MLTestResults['testCRMSE'].append(testCRMSE)
    MLTestResults['trainMAE'].append(trainMAE)
    MLTestResults['testSMAE'].append(testSMAE)
    MLTestResults['testCMAE'].append(testCMAE)
    MLTestResults['trainNLPD'].append(trainNLPD)
    MLTestResults['testSNLPD'].append(testSNLPD)
    MLTestResults['testCNLPD'].append(testCNLPD)
    MLTestResults['noise'].append(noise)

# save results
MLTestResults = pd.DataFrame(MLTestResults)
MLTestResults.to_csv(MLTestResultsPath, sep="\t", index=False)

# Train and save final models

In [ ]:
# path to storing models
modelsPath = projectDir/'ML'/'models'
modelsPath.mkdir(parents=True, exist_ok=True)

# storing trained model and model information
for depVar, noiseFloor, yTrain in [('Cd', noiseFloorDict['Cd'], yCdTrain), ('Cl', noiseFloorDict['Cl'], yClTrain)]:
    model, likelihood, noise = GPR(kernel, ARD, noiseFloor, XTrain, yTrain, returnModel = True)

    torch.save({
        'modelStateDict': model.state_dict(),
        'likelihoodStateDict': likelihood.state_dict(),
        'kernel': kernel,
        'ARD': ARD,
        'noiseConstraintFloor': float(noiseFloorDict[depVar]),
        'learnedNoise': noise,
        'XMin': XMin,
        'XMax': XMax,
        'XTrain': XTrain,
        'yTrain': yTrain,
        }, modelsPath/f'{depVar}GPRFinal.pth')